# Optional notebook - PCA, and what compression actually costs

**Outside the twelve sessions** · about 90 minutes, self-paced

---

## Learning objectives

By the end of this notebook you will be able to:

1. Say what PCA computes, and what question it answers. `[CLO1]`
2. Demonstrate why PCA without scaling answers a different question, and name
   which question. `[CLO6]`
3. Choose a number of components from explained variance, and state what that
   choice does not guarantee. `[CLO5]`
4. Read a component's loadings, and say what a loading is not. `[CLO2]`
5. Test whether dimensionality reduction helps a downstream model, rather than
   assuming it does. `[CLO5]`

## Prerequisites

Sessions 3, 4 and 6. Session 11 as well if you want §7, which uses the
sentence embeddings from that session.

## Why this notebook exists

Session 11 asks whether a **learned** representation beats a hand-made one. It
compares bag-of-words against a pretrained encoder and finds, on our data, that
the simple one wins.

There is a step missing from that comparison, and this notebook is it. PCA is
the classical middle: a representation that is **learned from the data but
without a target and without a neural network**. It is nearly a century old, it
takes one line, and it is the first thing anyone should try before reaching for
something larger.

It is also the most over-sold technique in this course's syllabus. You will be
told PCA "reduces noise", "speeds things up" and "improves models". We will
measure the third claim on our own data, twice, and it will not survive.

## §1 - What PCA computes

Take a cloud of points in $p$ dimensions. PCA finds a new set of axes for it:

- the **first** axis points along the direction of greatest variance;
- the **second** points along the greatest variance *perpendicular to the
  first*; and so on.

Nothing is discarded at this stage. It is a **rotation**: $p$ new axes for $p$
old ones, holding the same cloud. The reduction happens afterwards, when you
keep the first $k$ axes and throw the rest away.

Two things follow immediately, and both matter more than the algebra:

1. **PCA never sees the target.** It is unsupervised. It maximises variance in
   the features, and variance in the features is not the same thing as
   usefulness for predicting a target. Most of §6 is that sentence, measured.
2. **Variance depends on units.** A column measured in thousands has more
   variance than the same quantity measured in millions, so PCA on unscaled
   data is answering a question about your units.

### PCA as a rotation

![PCA rotation](../assets/diagrams/s11_representation/05.09-PCA-rotation.png)

On the left, the data in its original axes with the two principal directions
drawn on as vectors: one long, one short. On the right, the same points after
the rotation, now lying along those directions.

The right-hand picture is the whole idea. The cloud has not changed shape, it
has been described from a better angle - and because almost all of the spread
is now along one axis, dropping the other loses little. "Little" is doing a lot
of work in that sentence, and the rest of this notebook is about how little.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import PATHS, SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

# Session 6's scope, unchanged, so every number here is comparable with the ones
# from Sessions 6, 7 and 11.
df = load_raw()
train, _ = split_by_host(df)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float)
)
work = train[train["price_num"].notna()]
work = work[work["price_num"].between(10, 2000)]
work = work[work["minimum_nights"] <= 3].copy()
y = np.log1p(work["price_num"]).to_numpy()
groups = work["host_id"]

NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude",
           "longitude", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]

# Materialised once, so every comparison below is on identical folds and the
# paired tests in §6 and §7 are legitimate.
FOLDS = list(GroupKFold(n_splits=5).split(work, y, groups=groups))


def preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   min_frequency=20,
                                                   sparse_output=False))]),
         CATEGORICAL),
    ])


X = preprocessor().fit_transform(work)
print(f"listings        : {len(work):,}")
print(f"design matrix   : {X.shape[0]:,} x {X.shape[1]}  "
      f"({len(NUMERIC)} numeric + one-hot categoricals)")

## §2 - The scaling trap, in its sharpest form

In Session 3 scaling was about fairness between features in a distance. Here it
is more direct: PCA maximises **variance**, and variance is measured in the
units the column happens to be recorded in. So an unscaled PCA finds the
column with the biggest numbers and calls it the first principal component.

Run it both ways on the numeric columns alone and read the loadings.

In [ ]:
# Numeric columns only, so the loadings can be printed against names.
X_num = work[NUMERIC].fillna(work[NUMERIC].median()).to_numpy(dtype=float)

for label, matrix in (("UNSCALED", X_num),
                      ("SCALED  ", StandardScaler().fit_transform(X_num))):
    p = PCA().fit(matrix)
    # components_[0] is the first principal axis: one weight per original
    # column, and those weights are what "loadings" means.
    loadings = pd.Series(p.components_[0], index=NUMERIC).round(3)
    print(f"{label}  PC1 explains {p.explained_variance_ratio_[0]:.1%} "
          f"of the variance")
    # Sorted by magnitude, because the sign of a component is arbitrary - PCA
    # could return the exact opposite vector and it would mean the same thing.
    print(loadings.reindex(loadings.abs().sort_values(ascending=False).index)
          .to_string(), "\n")

### Read those two blocks again

**Unscaled**, the first component explains 67% of the variance and it is
`number_of_reviews` with a loading of **0.95**. Everything else is rounded to
zero. That component is not a summary of a listing; it is the review count
wearing a hat, and it is first only because review counts run into the
hundreds while bedroom counts run to about five.

**Scaled**, the first component explains 39% and the loadings are 0.50 on
`bedrooms`, 0.49 on `accommodates`, 0.49 on `beds` and 0.45 on `bathrooms`,
with 0.27 on `calculated_host_listings_count` and effectively nothing on the
rest. That is a **size** factor: one number describing how big a property is,
discovered without being told that those four columns measure related things.
The 0.27 is worth noticing too - larger properties tend to belong to hosts who
own more of them, which is a fact about this market rather than about geometry.

The second is worth having. The first is an artifact of the recording units.
Same data, same function call, and the only difference is one line of
preprocessing.

## §3 - Explained variance, and choosing k

`explained_variance_ratio_` gives the share of total variance each component
carries. They are sorted descending and they sum to 1. The usual way to choose
$k$ is to take the cumulative sum and stop at some threshold.

This is a weaker procedure than it looks, for a reason §6 makes concrete.

In [ ]:
pca_full = PCA().fit(X)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)

print(f"components in total: {len(cumulative)}")
for threshold in (0.5, 0.8, 0.9, 0.95, 0.99):
    # searchsorted finds the first position where the cumulative sum reaches the
    # threshold; +1 converts a 0-based index into a count of components.
    k = int(np.searchsorted(cumulative, threshold)) + 1
    print(f"  {threshold:.0%} of the variance needs {k:3d} components "
          f"({k / len(cumulative):.0%} of them)")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
# Left: the scree plot. The first component dominates and the rest decay - there
# is no sharp elbow here either, which is the same shape of disappointment as
# the k-means elbow in the clustering notebook.
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20],
            color="steelblue")
axes[0].set(title="scree: variance carried by each component",
            xlabel="component", ylabel="share of variance")
# Right: the cumulative version, which is what the threshold is read off.
axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", ms=3,
             color="darkorange")
for threshold in (0.9, 0.95):
    axes[1].axhline(threshold, color="grey", ls=":", lw=1)
axes[1].set(title="cumulative variance", xlabel="components kept",
            ylabel="share of variance")
plt.tight_layout()
plt.show()

## §4 - Reading a component, and what a loading is not

A component is a **weighted sum of the original columns**. The weights are the
loadings, and reading them is how you find out what a component means, if it
means anything.

Three cautions, all of which are routinely ignored:

1. **The sign is arbitrary.** PCA could return the exact negative of a
   component and it would be the same decomposition. Never interpret the
   direction of a loading on its own, only the pattern across columns.
2. **A component is not a feature you chose.** It is a mixture of all of them,
   so "component 3 is important" cannot be reported to a client. It has to be
   unpacked back into columns first, and sometimes it does not unpack into
   anything nameable.
3. **Naming a component is a hypothesis.** Calling PC1 "size" because four
   size-ish columns load on it is a reading, not a result. It is the same move
   as naming a cluster, and it needs the same caution.

In [ ]:
# TODO: Look at the first four components of the scaled numeric PCA.
#
#   1. Fit PCA on the scaled numeric columns.
#   2. Put the loadings in a DataFrame: components as columns, features as rows.
#   3. For each component, print the two features with the largest absolute
#      loading.
#   4. Try to name each component in a few words - and mark the ones you cannot
#      name honestly.

## §5 - What PCA is genuinely good at: looking

Before testing the claim that PCA improves models, here is the use that is not
in dispute. A 63-dimensional design matrix cannot be plotted. Its first two
principal components can.

In [ ]:
projected = PCA(n_components=2, random_state=SEED).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
# Coloured by room type: a category the PCA never saw, because PCA is
# unsupervised and was given no labels at all.
for room in ["Entire home/apt", "Private room", "Hotel room", "Shared room"]:
    mask = (work["room_type"] == room).to_numpy()
    if mask.sum() < 30:
        continue
    axes[0].scatter(projected[mask, 0], projected[mask, 1], s=6, alpha=0.35,
                    label=f"{room} (n={mask.sum():,})")
axes[0].set(title="first two components, coloured by room type",
            xlabel="PC1", ylabel="PC2")
axes[0].legend(fontsize=7, markerscale=2)

# Coloured by the target instead. If price varied smoothly along a direction in
# this plane, that would be a useful thing to know before modelling.
sc = axes[1].scatter(projected[:, 0], projected[:, 1], c=y, s=6, alpha=0.5,
                     cmap="viridis")
axes[1].set(title="the same points, coloured by log price", xlabel="PC1")
plt.colorbar(sc, ax=axes[1], label="log1p(price)")
plt.tight_layout()
plt.show()

print(f"these two components carry "
      f"{pca_full.explained_variance_ratio_[:2].sum():.1%} of the variance, "
      f"so about {1 - pca_full.explained_variance_ratio_[:2].sum():.0%} of it is "
      "not on this page at all")

Two things to take from the picture, and the second is the important one.

The room types separate visibly, and PCA was never told they existed. That is
a real result: the structure was in the features, and the rotation exposed it.

But these two components carry 46% of the variance, which means **54% of it is
not on the page at all**. Two points that look adjacent may be far apart in the
61 directions that were dropped. A 2-D PCA plot is a shadow, and a shadow can
make two people standing well apart look like one person. Never conclude "there
are three groups" from a scatter of two components.

## §6 - Does PCA improve the model?

This is the claim worth testing, because it is the one most often asserted.
The setup is a controlled comparison: identical folds, identical
preprocessing, identical estimator. The only change is a `PCA` step inserted
between them.

In [ ]:
# TODO: Predict first.
#
#   Ridge on all 63 features scores R2 = 0.607.
#   With PCA down to 10 components, I predict R2 = ____
#   The smallest k that matches the full model: ____
#
# Then run the sweep, and use a paired t-test against the full model rather than
# comparing the means by eye.

## §7 - The same test on the Session 11 embeddings

The tabular matrix had 63 columns, which is small. The argument for PCA is
usually made about genuinely high-dimensional data, where it should have more
room to help. Session 11's sentence embeddings give us 384 dimensions per
listing, so the claim can be tested where it is supposed to hold.

## §8 - So what is PCA for?

Everything above, in one table.

| claim | what we measured |
|---|---|
| "PCA summarises your features" | only after scaling. Unscaled, PC1 was 95% one column |
| "keep 90% of the variance" | 11 of 63 components, and keeping them cost 0.088 R2 |
| "PCA improves models" | no. Significantly worse at every real compression, twice |
| "PCA lets you see your data" | **yes**, and this is the use that survived |
| "components are interpretable" | sometimes. One of our four was honestly nameable |

So the honest summary is narrow, and worth having:

> PCA is a **diagnostic and a visualisation tool**. It tells you how much
> redundancy is in your feature set, it lets you put a 63-dimensional matrix on
> a page, and it occasionally exposes a structure you did not know was there.
> It is not a preprocessing step that makes models better, and on both of our
> datasets it made them measurably worse.

Where PCA genuinely does pay, and we did not test any of these:

- when $p$ is comparable to or larger than $n$, and the model cannot be fitted
  at all otherwise;
- when fitting cost is the binding constraint rather than accuracy;
- when the downstream method is distance-based and suffering in high
  dimensions;
- as a **noise filter**, reconstructing from a few components to discard what
  the tail was carrying.

None of those conditions holds for us: 6,302 rows against 63 columns, a model
that fits in under a second, and a linear estimator. Which is the point. The
technique is fine. The unconditional recommendation is what fails.

### The link back to Session 11

Session 11 found that a pretrained encoder did not beat bag-of-words on this
data. This notebook adds the row above it: a classical unsupervised
representation does not beat the raw features either. Both results say the same
thing in different vocabulary - **a representation has to earn its place
against the simpler thing, and most of the time, on ordinary tabular data, it
does not.**

## §9 - Knowledge check

From memory, then check.

1. PCA is unsupervised. Give the one-sentence reason that explains why it can
   discard exactly the direction your model needed.
2. Your first component has a loading of 0.95 on one column. What went wrong,
   and what is the fix?
3. You keep 90% of the variance and the model gets worse. Is that a
   contradiction?
4. A colleague reports "PC2 is important for predicting churn". What do you ask
   them?
5. Name one situation where PCA would genuinely have helped on this dataset.